In [ ]:
!pip install tensorflow

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout,
    BatchNormalization
)

import numpy as np

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications import MobileNet
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications import EfficientNetB0

# Data augmentation for training set
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1
)

# For validation and test sets, we only rescale the pixel values.
test_datagen = ImageDataGenerator(
    rescale=1./255
)

train_generator = train_datagen.flow_from_directory(
    r"..\DATA\train",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical"
)

val_generator = test_datagen.flow_from_directory(
    r"..\DATA\val",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical"
)

test_generator = test_datagen.flow_from_directory(
    r"..\DATA\test",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    shuffle=False
)

print(train_generator.class_indices)

model = Sequential()

# Block 1
model.add(Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)))
model.add(BatchNormalization())
model.add(MaxPooling2D((2,2)))

# Block 2
model.add(Conv2D(64, (3,3), activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D((2,2)))

# Block 3
model.add(Conv2D(128, (3,3), activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D((2,2)))

# Classification Head
model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(11, activation='softmax'))

# Compile the model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=2
)

test_loss, test_accuracy = model.evaluate(test_generator)

print("Test Accuracy:", test_accuracy)


    

In [ ]:
# Transfer Learning with Pre-trained Models
def build_transfer_model(base_model, num_classes):

    base_model.trainable = False

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.5)(x)

    predictions = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=predictions)

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

# Define the number of classes and input shape
num_classes = 11
input_shape = (224,224,3)

# Load pre-trained models without the top classification layer
models = {

"VGG16": VGG16(weights="imagenet", include_top=False, input_shape=input_shape),

"ResNet50": ResNet50(weights="imagenet", include_top=False, input_shape=input_shape),

"MobileNet": MobileNet(weights="imagenet", include_top=False, input_shape=input_shape),

"InceptionV3": InceptionV3(weights="imagenet", include_top=False, input_shape=input_shape),

"EfficientNetB0": EfficientNetB0(weights="imagenet", include_top=False, input_shape=input_shape)

}

# Train each model and evaluate on validation set
best_accuracy = 0
best_model = None
best_model_name = ""

for name, base_model in models.items():

    print("\nTraining:", name)

    model = build_transfer_model(base_model, num_classes)

    history = model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=2
    )

    val_acc = max(history.history['val_accuracy'])

    print(name, "Validation Accuracy:", val_acc)

    if val_acc > best_accuracy:
        best_accuracy = val_acc
        best_model = model
        best_model_name = name

print("Fine-tuning model:", best_model_name)

for layer in best_model.layers[-20:]:
    layer.trainable = True

best_model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

best_model.fit(train_generator,
    validation_data=val_generator,
    epochs=2
)



In [ ]:
best_model.save(r"..\MODEL\best_fish_model.h5")

In [ ]:
from tensorflow.keras.models import load_model

model = load_model(r"..\MODEL\best_fish_model.h5")

print("Model loaded successfully")

model.summary()

In [ ]:
!pip install streamlit tensorflow pillow numpy